# PySpark left join

## Source data

The source comes from jupyter-pyspark/f1-sourcefiles

# Inner the data from the races.csv and seasons.csv


# Initalise a spark session

In [1]:
# Initalise a spark session
import os
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip
from pyspark.sql.functions import col

# Fix JAVA_HOME to your actual Java 21 path
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-21-openjdk-amd64"
os.environ["PYSPARK_SUBMIT_ARGS"] = "--packages io.delta:delta-spark_2.12:3.2.0 pyspark-shell"

# Build Spark session with Delta Lake support
builder = SparkSession.builder \
    .appName("DeltaLakeExample") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

spark = configure_spark_with_delta_pip(builder).getOrCreate()


26/04/28 23:03:08 WARN Utils: Your hostname, DESKTOP-OQT8U26 resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/04/28 23:03:08 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/robyip/projects/pyspark-deltalake/.venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/robyip/.ivy2/cache
The jars for the packages stored in: /home/robyip/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-3121a18b-78af-475f-a19b-e8902a65accc;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 270ms :: artifacts dl 15ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.2.0 from central in [default]
	io.delta#delta-storage;3.2.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0

# Load races.csv and seasons.csv into dataframes

I will filter out races from before 2021

In [2]:
# Load csv files into dataframes
#  Contains headers
# Infers schema



races = spark.read.csv("f1-sourcefiles/races.csv", header=True, inferSchema=True).filter("year >= 2022")


seasons = spark.read.csv("f1-sourcefiles/seasons.csv", header=True, inferSchema=True)

# Show the data type of the dataframes


In [4]:
# results 

races.printSchema()

# races

seasons.printSchema()

root
 |-- raceId: integer (nullable = true)
 |-- year: integer (nullable = true)
 |-- round: integer (nullable = true)
 |-- circuitId: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- date: date (nullable = true)
 |-- time: string (nullable = true)
 |-- url: string (nullable = true)
 |-- fp1_date: string (nullable = true)
 |-- fp1_time: string (nullable = true)
 |-- fp2_date: string (nullable = true)
 |-- fp2_time: string (nullable = true)
 |-- fp3_date: string (nullable = true)
 |-- fp3_time: string (nullable = true)
 |-- quali_date: string (nullable = true)
 |-- quali_time: string (nullable = true)
 |-- sprint_date: string (nullable = true)
 |-- sprint_time: string (nullable = true)

root
 |-- year: integer (nullable = true)
 |-- url: string (nullable = true)



# Use Alias in a RIGHT JOIN


In [11]:
# RIGHT join on same column name i.e. year


# Create and alias for a Dataframe just like table alias in SQL
races = races.alias("rcs")

seasons = seasons.alias("seas")


# RIGHT join on year 
joined_df = races.join(seasons, col("rcs.year") == col("seas.year"), how="right") \
    .select(col("rcs.raceId"), col("rcs.year").alias("race_year"), col("seas.year"), col("seas.url").alias("season_url") ) \
    .filter("seas.year > 2020")

joined_df.explain(mode="formatted")

joined_df.show(100)

== Physical Plan ==
AdaptiveSparkPlan (8)
+- Project (7)
   +- BroadcastHashJoin RightOuter BuildLeft (6)
      :- BroadcastExchange (3)
      :  +- Filter (2)
      :     +- Scan csv  (1)
      +- Filter (5)
         +- Scan csv  (4)


(1) Scan csv 
Output [2]: [raceId#17, year#18]
Batched: false
Location: InMemoryFileIndex [file:/home/robyip/projects/pyspark-deltalake/jupyter-pyspark/f1-sourcefiles/races.csv]
PushedFilters: [IsNotNull(year), GreaterThanOrEqual(year,2022), GreaterThan(year,2020)]
ReadSchema: struct<raceId:int,year:int>

(2) Filter
Input [2]: [raceId#17, year#18]
Condition : ((isnotnull(year#18) AND (year#18 >= 2022)) AND (year#18 > 2020))

(3) BroadcastExchange
Input [2]: [raceId#17, year#18]
Arguments: HashedRelationBroadcastMode(List(cast(input[1, int, false] as bigint)),false), [plan_id=472]

(4) Scan csv 
Output [2]: [year#71, url#72]
Batched: false
Location: InMemoryFileIndex [file:/home/robyip/projects/pyspark-deltalake/jupyter-pyspark/f1-sourcefiles/seasons.csv